# 🧬 PyHMMER Tool Example

This notebook demonstrates how to use **PyHMMER** for protein and nucleotide sequence homology search.

## 📖 What is PyHMMER?

[PyHMMER](https://pyhmmer.readthedocs.io/) is a Python binding to HMMER, a tool for sequence analysis using profile Hidden Markov Models (HMMs).

### Key Features:
- **HMMSearch** -- Search HMM profile(s) against a set of protein sequences
- **HMMScan** -- Search protein sequences against an HMM database (e.g. Pfam)
- **PHMMER** -- Protein sequence vs protein sequence search (like BLASTP with HMM sensitivity)
- **NHMMER** -- Nucleotide sequence vs nucleotide sequence search
- **JackHMMER** -- Iterative protein sequence search for expanded homolog detection

## 📥 Imports

## 📦 Shared Data Types

### **`PyHmmerConfig`**
Shared configuration used by hmmsearch, hmmscan, phmmer, and nhmmer.

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `evalue_threshold` | `float` | `10.0` | E-value reporting threshold for sequence level hits |
| `score_threshold` | `Optional[float]` | `None` | Score reporting threshold (overrides the E-value threshold) |
| `domain_evalue_threshold` | `float` | `10.0` | E-value reporting threshold for domain level hits |
| `domain_score_threshold` | `Optional[float]` | `None` | Score reporting threshold for domain level hits (overrides the domain E-value threshold) |

### **`PyJackhmmerConfig`**
Extends `PyHmmerConfig` with:

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `max_iterations` | `int` | `5` | Maximum number of jackhmmer search iterations |

### **`PyHmmerOutput`**
Shared output used by all five tools.

| Field | Type | Description |
|-------|------|-------------|
| `sequence_hits_df` | `Optional[pd.DataFrame]` | DataFrame with per-sequence hits (columns: `query_name`, `query_accession`, `query_description`, `query_idx`, `target_name`, `target_accession`, `target_description`, `evalue`, `score`, `bias`, `sum_score`, `reported`, `included`, `pvalue`, `num_domains`) |
| `domain_hits_df` | `Optional[pd.DataFrame]` | DataFrame with per-domain hits (columns: `query_name`, `query_accession`, `query_description`, `query_idx`, `target_name`, `target_accession`, `target_description`, `hmm_length`, `hmm_from`, `hmm_to`, `target_from`, `target_to`, `target_length`, `c_evalue`, `i_evalue`, `domain_score`, `domain_bias`, `domain_idx`, `env_from`, `env_to`, `envelope_score`, `domain_included`, `domain_reported`, `domain_pvalue`) |

In [1]:
from bio_programming_tools.tools.gene_annotation.pyhmmer import (
    run_pyhmmer_hmmsearch, PyHmmsearchInput, PyHmmerConfig,
    run_pyhmmer_hmmscan, PyHmmscanInput,
    run_pyhmmer_phmmer, PyPhmmerInput,
    run_pyhmmer_nhmmer, PyNhmmerInput,
    run_pyhmmer_jackhmmer, PyJackhmmerInput, PyJackhmmerConfig,
)
from pathlib import Path

## 📂 1. HMMSearch (HMM Profile vs Sequences)

HMMSearch searches one or more HMM profiles against a set of protein sequences
to find sequences that match the profile. Use this when you have a specific
protein family model (HMM) and want to identify members in your dataset.

This example searches a small HMM database (containing Pkinase, Barwin, IPK,
and 1-cysPrx_C domain profiles) against two proteins:
- **Expected match**: Human kinase (B2R9J0) -- contains a Pkinase domain
- **Expected no match**: Human hemoglobin alpha -- unrelated protein, no kinase domain

### API Reference

**`PyHmmsearchInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Target protein sequences to search (single string or list) |
| `hmm` | `str \| Path` | Path to HMM file containing one or more profile HMMs |

In [2]:
# Path to HMM file containing domain profiles (Pkinase, Barwin, IPK, 1-cysPrx_C)
hmm_path = Path("./data/example_hmms.hmm")

# Target protein sequences:
#   0: Human kinase (B2R9J0) — contains Pkinase domain, expect match
#   1: Human hemoglobin alpha — no kinase domain, expect no match
kinase_seq = (
    "MLEPLPCWDAAKDLKEPQCPPGDRVGVQPGNSRVWQGTMEKAGLAWTRGTGVQSEGTWES"
    "QRQDSDALPSPELLPQDQDKPFLRKACSPSNIPAVIITDMGTQEDGALEETQGSPRGNLP"
    "LRKLSSSSASSTGFSSSYEDSEEDISSDPERTLDPNSAFLHTLDQQKPRVSKSWRKIKNM"
    "VHWSPFVMSFKKKYPWIQLAGHAGSFKAAANGRILKKHCESEQRCLDRLMVDVLRPFVPA"
    "YHGDVVKDGERYNQMDDLLADFDSPCVMDCKMGIRTYLEEELTKARRKPSLRKDMYQKMI"
    "EVDPEAPTEEEKAQRAVTKPRYMQWRETISSTATLGFRIEGIKKEDGTVNRDFKKTKTRE"
    "QVTEAFREFTKGNHNILIAYRDRLKAIRTTLEVSPFFKCHEVIGSSLLFIHDKKEQAKVW"
    "MIDFGKTTPLPEGQTLQHDVPWQEGNREDGYLSGLNNLVDILTEMSQDAPLA"
)

hemoglobin_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKL"
    "LSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
)

inputs = PyHmmsearchInput(
    hmm=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)
config = PyHmmerConfig(evalue_threshold=10.0)

hmmsearch_result = run_pyhmmer_hmmsearch(inputs, config)
print(f"Found {hmmsearch_result.num_sequence_hits} sequence hits")
print(f"Found {hmmsearch_result.num_domain_hits} domain hits")

if hmmsearch_result.sequence_hits_df is not None:
    print("\nSequence hits (target 0=kinase, 1=hemoglobin):")
    display(hmmsearch_result.sequence_hits_df[["query_name", "target_name", "evalue", "score"]])

Found 2 sequence hits
Found 2 domain hits

Sequence hits (target 0=kinase, 1=hemoglobin):


,query_name,target_name,evalue,score
0,Barwin,0,2.083643e-05,11.688442
1,IPK,0,6.497567e-50,156.782852


## 🗄️ 2. HMMScan (Sequences vs HMM Database)

HMMScan searches protein sequences against an HMM database to identify domains
and protein families within the query sequences. This is the reverse of
HMMSearch -- instead of "which sequences match this profile?", HMMScan asks
"which profiles (domains) are in this sequence?"

This example scans the same two proteins against the HMM database containing
four domain profiles (Pkinase, Barwin, IPK, 1-cysPrx_C) to annotate their
domain architecture.

### API Reference

**`PyHmmscanInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Query protein sequences to search (single string or list) |
| `hmm_db` | `str \| Path` | Path to HMM database file |

In [3]:
# Scan proteins against the HMM database to identify their domains
inputs = PyHmmscanInput(
    hmm_db=str(hmm_path),
    sequences=[kinase_seq, hemoglobin_seq],
)
config = PyHmmerConfig(evalue_threshold=10.0)

hmmscan_result = run_pyhmmer_hmmscan(inputs, config)
print(f"Found {hmmscan_result.num_sequence_hits} sequence hits")
print(f"Found {hmmscan_result.num_domain_hits} domain hits")

if hmmscan_result.domain_hits_df is not None:
    print("\nDomain annotations (sequence 0=kinase, 1=hemoglobin):")
    display(hmmscan_result.domain_hits_df[["query_name", "target_name", "c_evalue", "domain_score"]])

Found 2 sequence hits
Found 2 domain hits

Domain annotations (sequence 0=kinase, 1=hemoglobin):


,query_name,target_name,c_evalue,domain_score
0,0,IPK,9.610520e-50,156.228363
1,0,Barwin,3.906320e-05,10.807888


## 🔍 3. PHMMER Search (Sequence vs Sequence)

PHMMER searches query protein sequences against target sequences by building
a temporary HMM profile on-the-fly from the query. Similar to BLASTP but uses
profile HMM methods for improved sensitivity to distant homologs.

This example uses human hemoglobin alpha (82 residues) as the query and
three targets that demonstrate different match levels:
- **Strong match**: Mouse hemoglobin alpha -- close ortholog (~85% identity)
- **Partial match**: Human hemoglobin beta -- same globin family (~40% identity)
- **No match**: GFP fragment -- completely unrelated protein

### API Reference

**`PyPhmmerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Query protein sequences (single string or list) |
| `target_sequences` | `List[str]` | Target protein sequences to search against |

In [4]:
# Query: human hemoglobin alpha subunit (82 residues)
query_seq = (
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
    "GSAQVKGHGKKVADALTNAVAHVDDMPNALSA"
)

# Three targets at different match levels:
#   0: Mouse hemoglobin alpha  — close ortholog, expect strong match
#   1: Human hemoglobin beta   — same globin family, expect partial match
#   2: GFP fragment            — unrelated protein, expect no match
target_seqs = [
    "MVLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSH"
    "GSAQVKGHGKKVADALASAAGHLDDLPGALSAL",
    "VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLST"
    "PDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGT",
    "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTT"
    "GKLPVPWPTLVTTLTYGVQCFSRYPDHMKQ",
]

phmmer_inputs = PyPhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)
config = PyHmmerConfig(evalue_threshold=10.0)

phmmer_result = run_pyhmmer_phmmer(phmmer_inputs, config)
print(f"Found {phmmer_result.num_sequence_hits} sequence hits")
print(f"Found {phmmer_result.num_domain_hits} domain hits")

if phmmer_result.sequence_hits_df is not None:
    print("\nSequence hits (target 0=mouse alpha, 1=human beta, 2=GFP):")
    display(phmmer_result.sequence_hits_df[["query_name", "target_name", "evalue", "score"]])

Found 2 sequence hits
Found 2 domain hits

Sequence hits (target 0=mouse alpha, 1=human beta, 2=GFP):


,query_name,target_name,evalue,score
0,0,0,1.615991e-47,146.934052
1,0,1,1.674752e-16,48.288265


## 🧬 4. NHMMER Search (Nucleotide vs Nucleotide)

Search nucleotide query sequences against nucleotide target sequences using profile HMM methods.

NHMMER is the nucleotide counterpart of PHMMER. It builds a temporary HMM profile from the query DNA/RNA sequence and scans target nucleotide sequences for homologous regions. This is useful for finding conserved non-coding elements, regulatory regions, or homologous genes at the DNA level.

### API Reference

**`PyNhmmerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Query nucleotide sequences (single string or list) |
| `target_sequences` | `List[str]` | Target nucleotide sequences to search against |

In [5]:
# Query: human hemoglobin alpha coding DNA (first 150 nt)
query_dna = (
    "ATGGTGCTGTCTCCTGCCGACAAGACCAACGTCAAGGCCGCCTGGGGTAAG"
    "GTCGGCGCGCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTC"
    "CTGTCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACCTGAGCCAC"
)

# Two targets:
#   0: Mouse hemoglobin alpha coding DNA — ortholog, expect strong match
#   1: GFP coding DNA fragment — unrelated, expect no match
target_dna = [
    "ATGGTGCTGTCTGAGGACAAGAGCAACATCAAGGCCGCCTGGGGTAAGATC"
    "GGCGGCCACGCTGGCGAGTATGGTGCGGAGGCCCTGGAGAGGATGTTCGCC"
    "TCCTTCCCCACCACCAAGACCTACTTCCCGCACTTCGACGTGAGCCAC",
    "ATGAGTAAAGGAGAAGAACTTTTCACTGGAGTTGTCCCAATTCTTGTTGAAT"
    "TAGATGGTGATGTTAATGGGCACAAATTTTCTGTCAGTGGAGAGGGTGAAGG"
    "TGATGCTACATACGGAAAGCTTACCCTTAAATTTATTTGCACTACTGGAAAA",
]

nhmmer_inputs = PyNhmmerInput(
    sequences=[query_dna],
    target_sequences=target_dna,
)
nhmmer_config = PyHmmerConfig(evalue_threshold=10.0)

nhmmer_result = run_pyhmmer_nhmmer(nhmmer_inputs, nhmmer_config)
print(f"Found {nhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {nhmmer_result.num_domain_hits} domain hits")

if nhmmer_result.sequence_hits_df is not None:
    print("\nSequence hits (target 0=mouse alpha DNA, 1=GFP DNA):")
    display(nhmmer_result.sequence_hits_df[["query_name", "target_name", "evalue", "score"]])

Found 1 sequence hits
Found 1 domain hits

Sequence hits (target 0=mouse alpha DNA, 1=GFP DNA):


,query_name,target_name,evalue,score
0,0,0,1.266534e-36,112.83876


## 🔄 5. JackHMMER Search (Iterative)

Run an iterative protein sequence search against a protein database.

JackHMMER performs multiple rounds of searching: it builds an HMM from initial hits, then uses that refined profile to search again in subsequent iterations. This iterative approach can detect more distant homologs than a single-pass search. The search converges when no new sequences are found between iterations.

### API Reference

**`PyJackhmmerInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Query protein sequences (single string or list) |
| `target_sequences` | `List[str]` | Target protein sequences to search against |

**`PyJackhmmerConfig`** extends **`PyHmmerConfig`** with:

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `max_iterations` | `int` | `5` | Maximum number of jackhmmer search iterations |

In [6]:
# Reuse the same hemoglobin query and targets from the PHMMER example
jackhmmer_inputs = PyJackhmmerInput(
    sequences=[query_seq],
    target_sequences=target_seqs,
)
jackhmmer_config = PyJackhmmerConfig(
    max_iterations=3,
    evalue_threshold=10.0,
)

jackhmmer_result = run_pyhmmer_jackhmmer(jackhmmer_inputs, jackhmmer_config)
print(f"Found {jackhmmer_result.num_sequence_hits} sequence hits")
print(f"Found {jackhmmer_result.num_domain_hits} domain hits")

if jackhmmer_result.sequence_hits_df is not None:
    print("\nSequence hits:")
    display(jackhmmer_result.sequence_hits_df[["query_name", "target_name", "evalue", "score"]])

Found 2 sequence hits
Found 2 domain hits

Sequence hits:


,query_name,target_name,evalue,score
0,0,0,1.209558e-53,167.117050
1,0,1,1.798936e-42,131.316879


## 💾 6. Export Results

Save the search results to files for downstream analysis.

### Supported formats:
- **CSV** -- Tabular format with sequence-level and domain-level hits in separate files
- **JSON** -- Structured data with metadata and all hit information

The exported results can be used for:
- Filtering and ranking homologous sequences by E-value or bit score
- Domain architecture analysis and annotation
- Input to downstream phylogenetic or functional analyses

In [7]:
output_dir = Path("./example_output")

hmmsearch_result.export(name="hmmsearch_results", export_path=output_dir, file_format="csv")
hmmscan_result.export(name="hmmscan_results", export_path=output_dir, file_format="csv")
phmmer_result.export(name="phmmer_results", export_path=output_dir, file_format="csv")
nhmmer_result.export(name="nhmmer_results", export_path=output_dir, file_format="csv")
jackhmmer_result.export(name="jackhmmer_results", export_path=output_dir, file_format="csv")
print(f"Results exported to {output_dir}")

Results exported to example_output
